# 04 — Pre-registered analysis
Runs only after `prereg.json` exists and **has been uploaded to OSF / committed** (notebook 00, step 5).

Per pool (one dataset with all available encoders of its domain): held-out probe logits under grouped folds where
group ids exist, CKA matrix, anchor and fixed base, every candidate's marginal utility under three fusion rules,
and all matched pairs. Across pools: estimands A–F, for all pools and for each pre-registered subset
(pathology, vision, text, image).

**Which runtime.** The GPU is used only for the linear probes (seconds per encoder). Everything after them is CPU
work; for pools with many classes (Food-101, CIFAR-100, Country211, Banking77) that is most of the time, and a GPU
runtime sits idle through it. A **CPU high-RAM runtime** runs the same code with the probes on CPU (slower, but no
GPU quota). Pools already analysed are reused, so you can stop and switch runtimes between pools at any time.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, sys
PROJECT = "/content/drive/MyDrive/TMLR_v3"
CODE = os.path.join(PROJECT, "code")
assert os.path.exists(os.path.join(CODE, "config.py")), "run 00_setup_and_prereg.ipynb first"
if CODE not in sys.path:
    sys.path.insert(0, CODE)
import importlib, config as CFG
importlib.reload(CFG)
print("Drive:", CFG.DRIVE, "| project:", CFG.PROJECT)

In [ ]:
import torch
print("probe device:", "cuda" if torch.cuda.is_available() else "cpu")
import prereg, analyse_all as A
for m in (prereg, A):
    importlib.reload(m)
DEVIATION_NOTE = None   # set to a sentence ONLY if you deliberately changed registered code/parameters
status = prereg.check(DEVIATION_NOTE)
status

## Progress reporting (timing only — changes no number)
After the probes, each pool computes the CKA matrix and then fits the secondary class-conditional fusion
(5 base fits + 5 per candidate, about 45 per pool). Those steps run on the **CPU** and printed nothing, which made a
101-class pool look frozen (measured on Food-101: ~70 s per fit, ~50 min per pool). This cell wraps them with
timers so every step is visible, and logs every fusion fit to `results/fusion_fit_log.csv`, including whether it
converged or stopped at the registered 500-iteration cap (report that rate in the paper). It only times, prints and
logs: results are byte-identical with or without it (tested), and it does not touch the registered files, so the
pre-registration check is unaffected.

In [ ]:
import time, fusion, run_pipeline as RP
for m in (fusion, RP):
    importlib.reload(m)
importlib.reload(A)

def _timed(fn, label):
    def w(*a, **k):
        t = time.time()
        out = fn(*a, **k)
        print(f"    {label}: {time.time() - t:5.1f}s", flush=True)
        return out
    return w

RP.pairwise_cka_matrix = _timed(RP.pairwise_cka_matrix, "pairwise CKA matrix")
_cc_fit = fusion.IdentifiedClassConditionalFusion.fit
_state = {"pool": None, "n": 0}
FIT_LOG = os.path.join(CFG.RESULTS_DIR, "fusion_fit_log.csv")
_run_pool = RP.run_pool

def _run_pool_named(pool_name, *a, **k):          # only records which pool is running
    _state["pool"], _state["n"] = pool_name, 0
    return _run_pool(pool_name, *a, **k)

def _fit_timed(self, Zs, y):
    t = time.time()
    out = _cc_fit(self, Zs, y)
    _state["n"] += 1
    sec, it = time.time() - t, int(self.res_.nit)
    conv = bool(self.res_.success) and it < self.max_iter
    print(f"    class-conditional fusion fit {_state['n']:3d}: K={len(Zs)} C={Zs[0].shape[1]} "
          f"n={len(y)}  {sec:5.1f}s  ({it} iters{'' if conv else ', HIT ITERATION CAP'})", flush=True)
    new = not os.path.exists(FIT_LOG)
    with open(FIT_LOG, "a") as fh:
        if new:
            fh.write("pool,fit,K,C,n,iters,max_iter,converged,seconds\n")
        fh.write(f"{_state['pool']},{_state['n']},{len(Zs)},{Zs[0].shape[1]},{len(y)},"
                 f"{it},{self.max_iter},{conv},{sec:.1f}\n")
    return out

RP.run_pool = _run_pool_named
fusion.IdentifiedClassConditionalFusion.fit = _fit_timed
print("progress reporting on")

## Run every pool

In [ ]:
records, skipped = A.run_all(status)
print(f"\n{len(records)} pools analysed; skipped: {skipped}")

## Estimands across pools

In [ ]:
bat = A.battery(records, status)
md_text = A.write_all(records, bat, skipped)
from IPython.display import Markdown, display
display(Markdown(md_text))

## Figures

In [ ]:
from IPython.display import Image as IImage
for f in ("dose_response.png", "utility_vs_cka_and_competence.png"):
    display(IImage(os.path.join(CFG.RESULTS_DIR, "figures", f), width=700))

## Verification
Recomputes every estimand from the saved per-pool records and checks the result is identical.

In [ ]:
assert A.verify_determinism(records, bat), "battery is not reproducible from saved records"
import shutil
arch = shutil.make_archive(os.path.join(CFG.PROJECT, "results_bundle"), "zip", CFG.RESULTS_DIR)
print("results bundle:", arch)
print("files:", sorted(os.listdir(CFG.RESULTS_DIR)))